In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from category_encoders import TargetEncoder

# Fix random seed
def set_seed(seed=42):
    import random, torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed()

# Load data
df = pd.read_csv("train.csv")
y = df["Class"]
X = df.drop("Class", axis=1)

# Encode target
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Frequency encode Track Name
track_freq = X["Track Name"].value_counts().to_dict()
X["Track_Freq"] = X["Track Name"].map(track_freq)

# Target encode Track Name
track_target_encoder = TargetEncoder()
X["Track_Name_TE"] = track_target_encoder.fit_transform(X["Track Name"], y_encoded)

# Drop original Track Name
X = X.drop("Track Name", axis=1)

# Target encode Artist Name
artist_target_encoder = TargetEncoder()
X["Artist Name"] = artist_target_encoder.fit_transform(X["Artist Name"], y_encoded)

# Impute missing values
imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

# Replace outliers using Z-score with median
continuous_features = ['Popularity', 'danceability', 'energy', 'key', 'loudness', 'mode',
                       'speechiness', 'acousticness', 'instrumentalness', 'liveness',
                       'valence', 'tempo', 'duration_in min/ms', 'time_signature']

z_scores = np.abs((X_imputed[continuous_features] - X_imputed[continuous_features].mean()) /
                  X_imputed[continuous_features].std())

for feature in continuous_features:
    median_val = X_imputed[feature].median()
    X_imputed.loc[z_scores[feature] > 3, feature] = median_val

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

# Save preprocessed data
X_preprocessed_df = pd.DataFrame(X_scaled, columns=X.columns)
X_preprocessed_df["Target"] = y_encoded
X_preprocessed_df.to_csv("preprocessed_outlier_median_data.csv", index=False)
print("✅ Preprocessing done. Saved to preprocessed_outlier_median_data.csv.")


✅ Preprocessing done. Saved to preprocessed_outlier_median_data.csv.


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# Split the preprocessed data
X = X_scaled
y = y_encoded

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train SVM model
svm_model = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)  # You can change kernel to 'linear', 'poly', etc.
svm_model.fit(X_train, y_train)

# Make predictions
y_pred = svm_model.predict(X_test)

# Evaluate
print("✅ SVM Classification Report:")
print(classification_report(y_test, y_pred))
print("✅ Accuracy:", accuracy_score(y_test, y_pred))
print("✅ Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


✅ SVM Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       136
           1       0.56      0.68      0.62       286
           2       0.89      0.80      0.84       281
           3       1.00      0.91      0.95        78
           4       0.86      0.96      0.91        71
           5       0.92      0.85      0.88       262
           6       0.70      0.65      0.67       500
           7       0.99      0.98      0.99       103
           8       0.97      0.80      0.87       382
           9       0.89      0.79      0.83       531
          10       0.78      0.90      0.83       970

    accuracy                           0.82      3600
   macro avg       0.87      0.85      0.85      3600
weighted avg       0.83      0.82      0.82      3600

✅ Accuracy: 0.8155555555555556
✅ Confusion Matrix:
 [[134   0   0   0   2   0   0   0   0   0   0]
 [  0 195  24   0   0   1  36   0   0   2  28]
 [  0  31 22

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_encoded, test_size=0.2, random_state=42)

# Try different k values
best_k = None
best_accuracy = 0
best_model = None
best_predictions = None

for k in range(5, 21):  # Try k from 1 to 20
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    print(f"🔍 K = {k}, Accuracy = {acc:.4f}")

    if acc > best_accuracy:
        best_accuracy = acc
        best_k = k
        best_model = knn
        best_predictions = y_pred

# Final evaluation
print("\n✅ Best KNN Model Evaluation:")
print(f"Best k: {best_k}")
print(f"Accuracy: {best_accuracy:.4f}")
print("Classification Report:")
print(classification_report(y_test, best_predictions))
print("Confusion Matrix:")
print(confusion_matrix(y_test, best_predictions))


🔍 K = 5, Accuracy = 0.6561
🔍 K = 6, Accuracy = 0.6597
🔍 K = 7, Accuracy = 0.6678
🔍 K = 8, Accuracy = 0.6686
🔍 K = 9, Accuracy = 0.6767
🔍 K = 10, Accuracy = 0.6733
🔍 K = 11, Accuracy = 0.6828
🔍 K = 12, Accuracy = 0.6792
🔍 K = 13, Accuracy = 0.6803
🔍 K = 14, Accuracy = 0.6850
🔍 K = 15, Accuracy = 0.6861
🔍 K = 16, Accuracy = 0.6864
🔍 K = 17, Accuracy = 0.6867
🔍 K = 18, Accuracy = 0.6869
🔍 K = 19, Accuracy = 0.6828
🔍 K = 20, Accuracy = 0.6836

✅ Best KNN Model Evaluation:
Best k: 18
Accuracy: 0.6869
Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.93      0.92       136
           1       0.49      0.38      0.43       286
           2       0.75      0.75      0.75       281
           3       0.92      0.71      0.80        78
           4       0.79      0.92      0.85        71
           5       0.71      0.75      0.73       262
           6       0.55      0.55      0.55       500
           7       0.88      0.97      0.92

In [4]:
from sklearn.linear_model import Perceptron
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_encoded, test_size=0.2, random_state=42)

# Train Perceptron
perceptron = Perceptron(max_iter=1000, eta0=1.0, random_state=42)
perceptron.fit(X_train, y_train)

# Predict
y_pred = perceptron.predict(X_test)

# Evaluate
print("✅ Perceptron Evaluation:")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


✅ Perceptron Evaluation:
Accuracy: 0.5033333333333333
Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.89      0.92       136
           1       0.34      0.29      0.31       286
           2       0.54      0.79      0.64       281
           3       0.87      0.85      0.86        78
           4       0.88      0.20      0.32        71
           5       0.52      0.40      0.45       262
           6       0.38      0.30      0.33       500
           7       0.98      0.86      0.92       103
           8       0.36      0.86      0.51       382
           9       0.46      0.64      0.53       531
          10       0.76      0.31      0.44       970

    accuracy                           0.50      3600
   macro avg       0.64      0.58      0.57      3600
weighted avg       0.57      0.50      0.49      3600

Confusion Matrix:
 [[121   0   7   5   0   0   0   0   0   3   0]
 [  0  83  96   0   0  15  45   0  46   1  

In [5]:
from sklearn.linear_model import Perceptron
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.preprocessing import PolynomialFeatures

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_encoded, test_size=0.2, random_state=42)

# Variables to keep track of the best degree and its corresponding accuracy
best_degree = None
best_accuracy = 0
best_predictions = None

# Loop through polynomial degrees from 1 to 3 (you can expand this range as needed)
for degree in range(1, 4):  # Test degrees 1, 2, and 3
    # Apply polynomial features transformation
    poly = PolynomialFeatures(degree=degree, interaction_only=True, include_bias=False)
    X_train_poly = poly.fit_transform(X_train)
    X_test_poly = poly.transform(X_test)
    
    # Train Perceptron with class weights and increased iterations
    perceptron = Perceptron(max_iter=1000, eta0=1.0, class_weight='balanced', random_state=42)
    perceptron.fit(X_train_poly, y_train)
    
    # Predict and evaluate accuracy
    y_pred = perceptron.predict(X_test_poly)
    acc = accuracy_score(y_test, y_pred)
    
    # If the current degree has the best accuracy, update best_degree and best_accuracy
    if acc > best_accuracy:
        best_accuracy = acc
        best_degree = degree
        best_predictions = y_pred
        best_model = perceptron

# Final evaluation with the best polynomial degree
print(f"\n✅ Best Polynomial Degree: {best_degree}")
print(f"✅ Best Accuracy: {best_accuracy:.4f}")
print("✅ Classification Report:")
print(classification_report(y_test, best_predictions))
print("✅ Confusion Matrix:")
print(confusion_matrix(y_test, best_predictions))



✅ Best Polynomial Degree: 3
✅ Best Accuracy: 0.6844
✅ Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.93      0.90       136
           1       0.43      0.50      0.46       286
           2       0.65      0.77      0.70       281
           3       0.83      0.87      0.85        78
           4       0.68      0.92      0.78        71
           5       0.67      0.70      0.69       262
           6       0.56      0.50      0.53       500
           7       0.89      0.98      0.94       103
           8       0.72      0.76      0.74       382
           9       0.69      0.68      0.68       531
          10       0.78      0.68      0.73       970

    accuracy                           0.68      3600
   macro avg       0.71      0.75      0.73      3600
weighted avg       0.69      0.68      0.68      3600

✅ Confusion Matrix:
[[127   1   0   4   2   0   0   2   0   0   0]
 [  4 142  60   1   1   7  52   0   1   8 